In [ ]:
%pip install ipykernel
%pip install opencv-python 
%pip install matplotlib
%pip install matplotlib tk
%pip install plotly nbformat
%pip install pyqtgraph PyOpenGL PyQt5


In [14]:
import os
import time
from collections import deque
from dataclasses import dataclass

import cv2 as cv
import matplotlib
import numpy as np

import matplotlib.pyplot as plt

In [ ]:
@dataclass
class Config:
    camera_index: int = 0
    left_crop: int = 98             # columns removed from the left of each frame
    window_zoom: int = 4            # camera window size multiplier
    history_rows: int = 80          # time rows kept in the 3D map
    column_step: int = 4            # use every Nth image column
    update_seconds: float = 0.1     # time between 3D map updates
    map_size: tuple = (12, 8)       # 3D window size in inches
    max_value: int = 255


class Camera:
    def __init__(self, cfg):
        self.left_crop = cfg.left_crop
        self.capture = cv.VideoCapture(cfg.camera_index)
        if not self.capture.isOpened():
            raise RuntimeError("Could not open camera. Try another camera_index.")

    def read_gray(self):
        """Return the next frame as a cropped grayscale image, or None."""
        ok, frame = self.capture.read()
        if not ok:
            return None
        return cv.cvtColor(frame[:, self.left_crop:], cv.COLOR_BGR2GRAY)

    def release(self):
        self.capture.release()


class CameraView:
    def __init__(self, cfg):
        self.zoom = cfg.window_zoom

    def show(self, gray):
        colored = cv.applyColorMap(gray, cv.COLORMAP_INFERNO)
        cv.imshow("TC001", cv.resize(colored, None, fx=self.zoom, fy=self.zoom))

    @staticmethod
    def quit_requested():
        return cv.waitKey(1) & 0xFF in (ord("q"), 27)


class ThermalHistory:
    """Rolling rows of the hottest pixel in each image column."""

    def __init__(self, width, cfg):
        self.x = np.arange(0, width, cfg.column_step)
        self.step = cfg.column_step
        empty_row = np.zeros(len(self.x), np.float32)
        self.rows = deque([empty_row] * cfg.history_rows, maxlen=cfg.history_rows)
        self.pending = None

    def add_frame(self, gray):
        """Fold a frame into the pending row, keeping the hottest values seen."""
        row = gray.max(axis=0)[::self.step].astype(np.float32)
        self.pending = row if self.pending is None else np.maximum(self.pending, row)

    def commit(self):
        """Move the pending row into the history; the oldest row drops off."""
        self.rows.append(self.pending)
        self.pending = None

    def as_array(self):
        return np.array(self.rows)


class Map3D:
    def __init__(self, history, cfg):
        self.history = history
        self.max_value = cfg.max_value
        self.x, self.y = np.meshgrid(history.x, np.arange(cfg.history_rows))
        plt.ion()
        self.fig = plt.figure("Thermal 3D Map", figsize=cfg.map_size)
        self.ax = self.fig.add_subplot(111, projection="3d")

    def draw(self):
        """Redraw the surface. Returns False once the window has been closed."""
        self.ax.clear()
        self.ax.plot_surface(
            self.x, self.y, self.history.as_array(),
            cmap="inferno", vmin=0, vmax=self.max_value,
            linewidth=0, antialiased=False,
            rcount=self.x.shape[0], ccount=self.x.shape[1],
        )
        self.ax.set(xlabel="Image X", ylabel="Time", zlabel="Signal",
                    zlim=(0, self.max_value))
        self.ax.view_init(45, -70)
        plt.pause(0.001)
        return plt.fignum_exists(self.fig.number)


def run(cfg=Config()):
    camera = Camera(cfg)
    view = CameraView(cfg)
    history = map3d = None
    last_update = 0.0

    try:
        while True:
            gray = camera.read_gray()
            if gray is None:
                break

            if history is None:
                history = ThermalHistory(gray.shape[1], cfg)
                map3d = Map3D(history, cfg)

            view.show(gray)
            history.add_frame(gray)

            if time.monotonic() - last_update >= cfg.update_seconds:
                last_update = time.monotonic()
                history.commit()
                if not map3d.draw():
                    break

            if view.quit_requested():
                break
    finally:
        camera.release()
        cv.destroyAllWindows()
        plt.close("all")


if __name__ == "__main__":
    run()

In [ ]:
#Bounding box Code here and color bar 